In [1]:
import tensorflow as ts
from tensorflow.keras.layers import Input,Lambda,Dense,Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img
from tensorflow.keras.models import Sequential
import numpy as np
from glob import glob

In [2]:
IMAGE_SIZE=[224,224]


In [3]:
import os
import random
import shutil

# ===== BASE DATASET PATH =====
base_dir = "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset"

classes = ["covid", "normal", "pneumonia"]

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# ===== CREATE TARGET DIRECTORIES =====
for split in ["train", "val", "test"]:
    for cls in classes:
        os.makedirs(os.path.join(base_dir, split, cls), exist_ok=True)

# ===== SPLIT FUNCTION =====
def split_class_images(class_name):
    src_dir = os.path.join(base_dir, class_name)
    images = [img for img in os.listdir(src_dir) if not img.startswith(".")]
    random.shuffle(images)

    total = len(images)
    train_end = int(train_ratio * total)
    val_end = train_end + int(val_ratio * total)

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    for img in train_imgs:
        shutil.copy(os.path.join(src_dir, img),
                    os.path.join(base_dir, "train", class_name, img))

    for img in val_imgs:
        shutil.copy(os.path.join(src_dir, img),
                    os.path.join(base_dir, "val", class_name, img))

    for img in test_imgs:
        shutil.copy(os.path.join(src_dir, img),
                    os.path.join(base_dir, "test", class_name, img))

    print(f"✅ {class_name}: Train={len(train_imgs)}, Val={len(val_imgs)}, Test={len(test_imgs)}")

# ===== RUN FOR ALL CLASSES =====
for cls in classes:
    split_class_images(cls)

print("🎉 Train / Val / Test split completed successfully!")


✅ covid: Train=1619, Val=346, Test=348
✅ normal: Train=1619, Val=346, Test=348
✅ pneumonia: Train=1619, Val=346, Test=348
🎉 Train / Val / Test split completed successfully!


In [4]:
train_path='/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/train'
test_path='/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/test'

In [5]:
vgg16=VGG16(input_shape=IMAGE_SIZE+[3],weights='imagenet',include_top=False)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step


In [6]:
for layer in vgg16.layers:
    layer.trainable=False


In [7]:
folders=glob('train_path/*')


In [8]:
x=Flatten()(resnet.output)

NameError: name 'resnet' is not defined

In [9]:
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Model

# Use vgg16, NOT resnet
x = Flatten()(vgg16.output)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(3, activation="softmax")(x)   # covid, normal, pneumonia

model = Model(inputs=vgg16.input, outputs=output)


In [10]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

batch_size = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=batch_size,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    val_path,
    target_size=(224,224),
    batch_size=batch_size,
    class_mode="categorical"
)


Found 4835 images belonging to 3 classes.


NameError: name 'val_path' is not defined

In [12]:
train_path = "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/train"
val_path   = "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/val"
test_path  = "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/test"


In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

batch_size = 16#32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=batch_size,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    val_path,
    target_size=(224,224),
    batch_size=batch_size,
    class_mode="categorical"
)


Found 4835 images belonging to 3 classes.
Found 1031 images belonging to 3 classes.


In [15]:
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen
)


Epoch 1/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 708s 2s/step - accuracy: 0.7229 - loss: 0.7694 - val_accuracy: 0.8710 - val_loss: 0.3616
Epoch 2/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 879s 3s/step - accuracy: 0.7948 - loss: 0.5014 - val_accuracy: 0.9040 - val_loss: 0.3020
Epoch 3/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 1246s 4s/step - accuracy: 0.8196 - loss: 0.4758 - val_accuracy: 0.8885 - val_loss: 0.3013
Epoch 4/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 1217s 4s/step - accuracy: 0.8267 - loss: 0.4456 - val_accuracy: 0.8875 - val_loss: 0.2922
Epoch 5/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 1010s 3s/step - accuracy: 0.8325 - loss: 0.4373 - val_accuracy: 0.9040 - val_loss: 0.2721
Epoch 6/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 1028s 3s/step - accuracy: 0.8422 - loss: 0.4171 - val_accuracy: 0.9059 - val_loss: 0.2790
Epoch 7/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 3644s 12s/step - accuracy: 0.8412 - loss: 0.4219 - val_accuracy: 0.9079 - val_loss: 0.2632
Epoch 8/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 936s 3s/step - accuracy: 0.8449 - loss: 0.4025 - va

In [16]:
test_gen = val_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

model.evaluate(test_gen)


Found 1036 images belonging to 3 classes.
65/65 ━━━━━━━━━━━━━━━━━━━━ 99s 2s/step - accuracy: 0.9035 - loss: 0.2490


[0.24904273450374603, 0.9034749269485474]